# 入出力：CSV の要約と保存

対応する本文: [io.tex](../chapters/io.tex)。各問を「考え方 → 実装 → 確認」の順で解説する。上から全セルを実行すると、入力の作成から結果の検証までを再現できる。生成物は専用の出力先へ保存する。

In [1]:
import sys
from pathlib import Path

# リポジトリ直下と notebooks/ のどちらから起動しても同じ場所を使う。
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "main.tex").exists() and (p / "chapters").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("教材のフォルダー内で Notebook を起動してください。")
WORK = ROOT / "notebooks" / "_outputs" / "04_io"
WORK.mkdir(parents=True, exist_ok=True)
print("Python", sys.version.split()[0])
print("出力先:", WORK.relative_to(ROOT))

Python 3.12.14
出力先: notebooks/_outputs/04_io


## 問1：CSV を読み、JSON に要約する

引用符内にカンマがある地名も `csv.DictReader` なら1つのフィールドとして読める。空欄は空文字列として得られる。入力が未整列でも最初と最後のイベント時刻を正しく求められるよう、UTC の日時へ変換して最小値と最大値を取る。

実データへ適用する場合は `CSV_PATH` を取得済みカタログへ変更する。人工データは別名で作成する。

In [2]:
CSV_PATH = WORK / "usgs_example.csv"
CSV_PATH.write_text('time,latitude,longitude,mag,place\n2024-01-01T00:00:00.000Z,35.0,140.0,3.2,"Offshore, Japan"\n2024-01-01T00:05:00.000Z,36.0,141.0,,Japan\n2024-01-01T00:12:30.000Z,34.5,139.5,2.8,Tokyo\n', encoding="utf-8")

181

In [3]:
import csv
import json
import math
from datetime import datetime

with CSV_PATH.open(encoding="utf-8", newline="") as f:
    rows = list(csv.DictReader(f))
times = [datetime.fromisoformat(row["time"]) for row in rows]
valid_magnitudes = []
skipped = {"empty": 0, "invalid": 0, "nonfinite": 0}
for row in rows:
    text = row["mag"].strip()
    if not text:
        skipped["empty"] += 1
        continue
    try:
        value = float(text)
    except ValueError:
        skipped["invalid"] += 1
        continue
    if not math.isfinite(value):
        skipped["nonfinite"] += 1
        continue
    valid_magnitudes.append(value)
summary = {
    "n_events": len(rows),
    "first_time": min(times).isoformat() if times else None,
    "last_time": max(times).isoformat() if times else None,
    "mag_min": min(valid_magnitudes) if valid_magnitudes else None,
    "mag_max": max(valid_magnitudes) if valid_magnitudes else None,
    "skipped_magnitudes": skipped,
}
results = WORK / "results"
results.mkdir(parents=True, exist_ok=True)
with (results / "usgs_summary.json").open("w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2, allow_nan=False)
with (results / "usgs_summary.json").open(encoding="utf-8") as f:
    restored = json.load(f)
assert restored == summary
assert (summary["n_events"], summary["mag_min"], summary["mag_max"]) == (3, 2.8, 3.2)
assert summary["first_time"] == "2024-01-01T00:00:00+00:00"
assert summary["last_time"] == "2024-01-01T00:12:30+00:00"
assert rows[0]["place"] == "Offshore, Japan"
print(json.dumps(summary, ensure_ascii=False, indent=2))

{
  "n_events": 3,
  "first_time": "2024-01-01T00:00:00+00:00",
  "last_time": "2024-01-01T00:12:30+00:00",
  "mag_min": 2.8,
  "mag_max": 3.2,
  "skipped_magnitudes": {
    "empty": 1,
    "invalid": 0,
    "nonfinite": 0
  }
}


`parents=True` は親も作り、`exist_ok=True` は既存のフォルダーを許す。`indent=2` は字下げを2文字にする。`None` は JSON の `null` になる。`allow_nan=False` によって JSON の標準外の数値を保存しようとすると失敗する。

## 問2：中間データを Pickle へ保存する

ここでは元の CSV の文字列を保存する。型を変換する場合はその規則も記録する。`wb` と `rb` の `b` はバイナリモードを表す。Pickle の復元は任意のコードを実行し得るため、自分が保存したファイルだけを読み戻す。

In [4]:
import pickle

preview = [{key: row[key] for key in ["time", "latitude", "longitude", "mag"]} for row in rows[:200]]
path = results / "usgs_preview.pkl"
with path.open("wb") as f:
    pickle.dump(preview, f)
with path.open("rb") as f:
    restored_preview = pickle.load(f)
assert restored_preview == preview and len(restored_preview) == 3
print(len(restored_preview), restored_preview[0])

3 {'time': '2024-01-01T00:00:00.000Z', 'latitude': '35.0', 'longitude': '140.0', 'mag': '3.2'}


## 問3：空欄・変換失敗・非有限値を分けて記録する

`float("NaN")` と `float("inf")` は例外にならない。有限なマグニチュードだけを集計する場合は `math.isfinite` が必要である。今回の入力では空欄が1件、それ以外は0件となる。

In [5]:
assert skipped == {"empty": 1, "invalid": 0, "nonfinite": 0}
(results / "skipped_rows.txt").write_text(
    "\n".join(f"{key}: {value}" for key, value in skipped.items()) + "\n", encoding="utf-8")
print((results / "skipped_rows.txt").read_text(encoding="utf-8"))

empty: 1
invalid: 0
nonfinite: 0

